# White Wine Quality Prediction

## 1. Project overview

This beginner-level machine learning project uses physicochemical measurements of white wine to predict its quality score. This is a **regression** problem because the target (`quality`) is a numerical score.

**Project workflow:** explore the data, prepare features and target, create a train/test split, establish a baseline, train and compare regression models, evaluate the selected model, and interpret its feature importance.

The dataset is the UCI white wine quality dataset. The CSV uses semicolons as separators.

## 2. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("../data/winequality-white.csv", sep=";")
df.head()

## 3. Exploratory data analysis (EDA)

First, inspect the dataset dimensions, column types, missing values, duplicates and basic statistics.

In [ ]:
print("Dataset shape:", df.shape)
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df.describe()

### Target distribution

The target is `quality`. Understanding its distribution helps us see which scores are common and which are rare.

In [ ]:
df["quality"].value_counts().sort_index()

In [ ]:
df["quality"].value_counts().sort_index().plot(
    kind="bar", xlabel="Quality score", ylabel="Number of wines",
    title="Distribution of White Wine Quality Scores"
)
plt.tight_layout()
plt.show()

### Explore relationships with quality

Check how alcohol content varies across quality scores and inspect correlations. Correlation describes association; it does not prove causation.

In [ ]:
df.boxplot(column="alcohol", by="quality", figsize=(9, 5))
plt.title("Alcohol Content by Wine Quality")
plt.suptitle("")
plt.xlabel("Quality score")
plt.ylabel("Alcohol content")
plt.tight_layout()
plt.show()

In [ ]:
correlations = df.corr(numeric_only=True)["quality"].drop("quality").sort_values(ascending=False)
correlations

In [ ]:
correlations.sort_values().plot(kind="barh", figsize=(8, 6))
plt.xlabel("Correlation with quality")
plt.title("Feature Correlations with Wine Quality")
plt.tight_layout()
plt.show()

### EDA observations

- The target scores are concentrated around the middle values, especially 5 and 6.
- Alcohol has the strongest positive linear correlation with quality in this dataset.
- Density and volatile acidity have negative correlations with quality.
- Correlation is useful for exploration, but a feature's correlation alone does not determine how useful it will be to a machine learning model.

The dataset contains duplicate rows. For this beginner project, we retain the original observations so the results remain consistent with the experiments performed. A more detailed project could investigate deduplication as a separate data-preparation decision.

## 4. Prepare features and target

`X` contains the input measurements and `y` contains the quality score we want to predict.

In [ ]:
X = df.drop(columns=["quality"])
y = df["quality"]

print("Features:", X.shape)
print("Target:", y.shape)

## 5. Split into training and test sets

The training set is used to fit models. The test set is used for a final evaluation after model selection. We use a fixed random state so the split is reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 6. Establish a baseline

A baseline model gives us a simple reference point. `DummyRegressor` predicts the mean quality score from the training set for every test example.

In [ ]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

print("Baseline MAE:", mean_absolute_error(y_test, baseline_predictions))
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test, baseline_predictions)))
print("Baseline R²:", r2_score(y_test, baseline_predictions))

## 7. Train and evaluate regression models

We begin with Linear Regression as a simple model, then try Decision Tree, Random Forest and Extra Trees. We use the same three metrics throughout:

- **MAE:** average absolute prediction error; lower is better.
- **RMSE:** gives larger errors more weight; lower is better.
- **R²:** measures improvement over predicting the target mean; higher is better.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_features="sqrt", random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=300, max_features="sqrt", random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R²": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
results_df.round(4)

## 8. Compare models with cross-validation

A single train/test split can give a result that depends on which examples happen to be in the test set. Cross-validation repeats training and validation across several folds of the training data. We use five folds and compare the models using the same splits.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

cv_results = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=kfold,
        scoring=scoring, n_jobs=-1
    )
    cv_results.append({
        "Model": name,
        "Mean MAE": -scores["test_mae"].mean(),
        "Mean RMSE": -scores["test_rmse"].mean(),
        "Mean R²": scores["test_r2"].mean()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("Mean R²", ascending=False)
cv_results_df.round(4)

## 9. Select and train the final model

Based on the experiments, Extra Trees performed best among the models tested. Its selected settings are 300 trees, unrestricted maximum depth, a minimum leaf size of 1 (the default), and `max_features="sqrt"`.

We now fit this model on the complete training set and evaluate it on the test set.

In [ ]:
final_model = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
final_r2 = r2_score(y_test, y_pred)

print("Test MAE:", final_mae)
print("Test RMSE:", final_rmse)
print("Test R²:", final_r2)

## 10. Examine predictions and residuals

A residual is the actual value minus the predicted value. Positive residuals mean the model predicted too low; negative residuals mean it predicted too high.

In [ ]:
comparison = pd.DataFrame({
    "Actual quality": y_test.to_numpy(),
    "Predicted quality": y_pred
})
comparison["Residual"] = comparison["Actual quality"] - comparison["Predicted quality"]
comparison.head(10)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "--")
plt.xlabel("Actual quality")
plt.ylabel("Predicted quality")
plt.title("Actual vs Predicted Wine Quality")
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test.to_numpy() - y_pred

plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=25, edgecolor="black")
plt.axvline(0, linestyle="--")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")
plt.tight_layout()
plt.show()

print("Mean residual:", residuals.mean())

## 11. Feature importance

Tree-based models provide impurity-based feature importance scores. These indicate how useful each feature was for splitting the trees. They do not show whether a feature increases or decreases quality, and they do not establish causation.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(8, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Extra Trees Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 12. Conclusion

In this project, we explored the white wine quality dataset and trained several regression models. We compared a simple baseline, Linear Regression, Decision Tree, Random Forest and Extra Trees. Extra Trees was the strongest of the models tested and was selected as the final model.

The final test metrics from the recorded experiment were approximately:

- **MAE:** 0.3598
- **RMSE:** 0.5603
- **R²:** 0.5947

Alcohol was the most important feature according to the model's impurity-based importance, followed by other physicochemical measurements. These are predictive relationships, not causal conclusions.

### Limitations

- Wine quality scores are concentrated around the middle of the scale, so the model has fewer examples of rare scores.
- Predictions are continuous decimal values even though the observed quality scores are integers.
- The model does not explain all variation in quality; other factors not present in the dataset may matter.
- During experimentation, the test-set results were consulted while comparing model configurations. Therefore, the final test metrics should be treated as an estimate rather than a completely untouched assessment. A future study could reserve a fresh holdout set or use nested cross-validation for model selection.

This project demonstrates a basic end-to-end regression workflow, from data exploration through model evaluation and interpretation.